### Private RAG 🕹️

#### 0. 환경설정

In [1]:
import torch

assert torch.cuda.is_available(), "GPU가 연결되지 않았습니다. 런타임 유형을 T4 GPU로 변경하세요."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU  : Tesla T4
VRAM : 15.6 GB


In [2]:
!pip install -q --only-binary=:all: tokenizers
!pip install -q --upgrade "transformers>=4.46,<4.50" "sentence-transformers>=3.2" "qdrant-client>=1.11" "accelerate>=0.34"

In [3]:
import importlib

for pkg in ["transformers", "sentence_transformers", "qdrant_client"]:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "버전 정보 없음")
        print(f"✅ {pkg:<22} {version}")
    except ImportError as e:
        print(f"❌ {pkg:<22} 임포트 실패: {e}")

✅ transformers           4.49.0
✅ sentence_transformers  5.7.0
✅ qdrant_client          버전 정보 없음


In [4]:
!pip show qdrant-client

Name: qdrant-client
Version: 1.19.1
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 


In [5]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")
print("Qdrant 클라이언트 정상 생성:", client)

Qdrant 클라이언트 정상 생성: <qdrant_client.qdrant_client.QdrantClient object at 0x78dccc46ae40>


---

#### 1. 근로기준법 문서 준비

In [6]:
# 문서 불러오기
import requests

url = "https://github.com/legalize-kr/legalize-kr/raw/refs/heads/main/kr/%EA%B7%BC%EB%A1%9C%EA%B8%B0%EC%A4%80%EB%B2%95/%EB%B2%95%EB%A5%A0(%EB%B2%95%EB%A5%A0).md"
response = requests.get(url)
response.encoding = "utf-8"

raw_text = response.text
print(f"응답 코드: {response.status_code}")
print(f"전체 길이: {len(raw_text)}자")
print(raw_text[:300])

응답 코드: 200
전체 길이: 59908자
---
제목: 근로기준법
법령MST: 286771
법령ID: '001872'
법령구분: 법률
법령구분코드: ''
소관부처:
- 고용노동부
공포일자: 2026-06-09
공포번호: '21784'
시행일자: 2027-06-10
법령분야: ''
상태: 시행
출처: https://www.law.go.kr/법령/근로기준법
첨부파일:
- 별표번호: '0000'
  별표가지번호: '00'
  별표구분: 별표
  제목: 신체장해등급과 재해보상표(제80조 관련)
  파일링크: https://www.law.go.kr/LSW/flDownload.do?


In [7]:
# 본문과 부칙 분리
# "00 부칙" 이전까지만 본문(조문)으로 취급
# 부칙은 개정 이력 모음이라 성격이 다르므로 따로 떼어둠
main_body, _, appendix = raw_text.partition("## 부칙")

print(f"본문 길이: {len(main_body)}자")
print(f"부칙 길이: {len(appendix)}자")

본문 길이: 42259자
부칙 길이: 17644자


In [8]:
# 조문 단위로 분리
import re

# "##### 제N조" 앞에서 자릅니다 (제N조의N 가지번호도 포함)
articles_raw = re.split(r"(?=##### 제\d+조)", main_body)
articles_raw = [a.strip() for a in articles_raw if a.strip().startswith("##### 제")]

chunks = []
for a in articles_raw:
    # 조항 제목 추출 "##### 제17조 (근로조건의 명시)"
    title_match = re.match(r"##### (제\d+조(?:의\d+)?(?:\s*\([^)]*\))?)", a)
    조항 = title_match.group(1) if title_match else "미상"

    # 앵커 링크 줄 [#...](#...) 제거
    body = re.sub(r"\[#.*?\]\(#.*?\)\n?", "", a)
    # 제목 줄(#####...) 제거해서 본문만 남김
    body = re.sub(r"^##### .*\n", "", body).strip()

    chunks.append({"조항": 조항, "text": body})

print(f"총 조문 개수: {len(chunks)}개")
print(f"\n예시 (제17조):")
example = [c for c in chunks if c["조항"].startswith("제17조")][0]
print(f"[{example['조항']}]")
print(example["text"])

총 조문 개수: 133개

예시 (제17조):
[제17조 (근로조건의 명시)]
**①** 사용자는 근로계약을 체결할 때에 근로자에게 다음 각 호의 사항을 명시하여야 한다. 근로계약 체결 후 다음 각 호의 사항을 변경하는 경우에도 또한 같다. <개정 2010.5.25>

  1\. 임금
  2\. 소정근로시간
  3\. 제55조에 따른 휴일
  4\. 제60조에 따른 연차 유급휴가
  5\. 그 밖에 대통령령으로 정하는 근로조건

**②** 사용자는 제1항제1호와 관련한 임금의 구성항목ㆍ계산방법ㆍ지급방법 및 제2호부터 제4호까지의 사항이 명시된 서면(「전자문서 및 전자거래 기본법」 제2조제1호에 따른 전자문서를 포함한다)을 근로자에게 교부하여야 한다. 다만, 본문에 따른 사항이 단체협약 또는 취업규칙의 변경 등 대통령령으로 정하는 사유로 인하여 변경되는 경우에는 근로자의 요구가 있으면 그 근로자에게 교부하여야 한다. <신설 2010.5.25, 2021.1.5>


In [9]:
print(f"총 조문: {len(chunks)}개")
print("가지번호 조문 예시:", [c["조항"] for c in chunks if "의" in c["조항"]][:10])

총 조문: 133개
가지번호 조문 예시: ['제2조 (정의)', '제3조 (근로조건의 기준)', '제4조 (근로조건의 결정)', '제5조 (근로조건의 준수)', '제7조 (강제 근로의 금지)', '제8조 (폭행의 금지)', '제9조 (중간착취의 배제)', '제10조 (공민권 행사의 보장)', '제13조 (보고, 출석의 의무)', '제14조 (법령 주요 내용 등의 게시)']


##### 법령 최고 조 번호는 제116조였지만,
##### 가지번호 조문(제43조의2, 제43조의3 처럼 "의N"이 붙은 추가 조문)이 많아서 133개로 늘어남

In [10]:
# 원문자 뒤 슬래시 삭제
for c in chunks:
    c["text"] = c["text"].replace("\\.", ".")

# 확인
example = [c for c in chunks if c["조항"].startswith("제17조")][0]
print(example["text"][:200])

# 삭제 조문 처리
deleted_articles = [c["조항"] for c in chunks if "삭제" in c["text"] and len(c["text"]) < 30]
print(f"삭제된 조문: {deleted_articles}")

**①** 사용자는 근로계약을 체결할 때에 근로자에게 다음 각 호의 사항을 명시하여야 한다. 근로계약 체결 후 다음 각 호의 사항을 변경하는 경우에도 또한 같다. <개정 2010.5.25>

  1. 임금
  2. 소정근로시간
  3. 제55조에 따른 휴일
  4. 제60조에 따른 연차 유급휴가
  5. 그 밖에 대통령령으로 정하는 근로조건

**②** 사
삭제된 조문: ['제35조', '제102조', '제102조의2', '제103조', '제105조']


In [11]:
chunks_clean = [c for c in chunks if not ("삭제" in c["text"] and len(c["text"]) < 30)]

print(f"전체 조문: {len(chunks)}개")
print(f"삭제 조문 제외 후: {len(chunks_clean)}개")

# 이후 단계부터는 chunks_clean을 사용
chunks = chunks_clean

전체 조문: 133개
삭제 조문 제외 후: 128개


In [12]:
# 확인하기
import random

sample = random.sample(chunks, 3)
for c in sample:
    print(f"[{c['조항']}]")
    print(c["text"][:150])
    print("-" * 50)

[제44조의2 (건설업에서의 임금 지급 연대책임)]
**①** 건설업에서 사업이 2차례 이상 「건설산업기본법」 제2조제11호에 따른 도급(이하 "공사도급"이라 한다)이 이루어진 경우에 같은 법 제2조제7호에 따른 건설사업자가 아닌 하수급인이 그가 사용한 근로자에게 임금(해당 건설공사에서 발생한 임금으로 한정한다)을 지급
--------------------------------------------------
[제76조 (안전과 보건)]
근로자의 안전과 보건에 관하여는 「산업안전보건법」에서 정하는 바에 따른다.

## 제6장의2 직장 내 괴롭힘의 금지 <신설 2019.1.15>
--------------------------------------------------
[제17조 (근로조건의 명시)]
**①** 사용자는 근로계약을 체결할 때에 근로자에게 다음 각 호의 사항을 명시하여야 한다. 근로계약 체결 후 다음 각 호의 사항을 변경하는 경우에도 또한 같다. <개정 2010.5.25>

  1. 임금
  2. 소정근로시간
  3. 제55조에 따른 휴일
  4. 제
--------------------------------------------------


In [13]:
# 청크 수 및 길이 확인
import numpy as np

lengths = [len(c["text"]) for c in chunks]

print(f"청크 수 : {len(chunks)}")
print(f"평균 청크 길이 : {np.mean(lengths):.0f}자")
print(f"최소 : {min(lengths)}자 / 최대 : {max(lengths)}자")
print(f"중앙값 : {np.median(lengths):.0f}자")

청크 수 : 128
평균 청크 길이 : 299자
최소 : 25자 / 최대 : 1613자
중앙값 : 190자


In [14]:
# chunks에 메타데이터가 제대로 달려있는지 확인
sample = chunks[10]
print(f"조항: {sample['조항']}")
print(f"text 앞부분: {sample['text'][:80]}")
print(f"딕셔너리 키: {sample.keys()}")

조항: 제11조 (적용 범위)
text 앞부분: **①** 이 법은 상시 5명 이상의 근로자를 사용하는 모든 사업 또는 사업장에 적용한다. 다만, 동거하는 친족만을 사용하는 사업 또는 사업장과
딕셔너리 키: dict_keys(['조항', 'text'])


In [15]:
eval_questions = [
    # 의미 질문
    ("근로계약서에 반드시 명시해야 하는 내용은 뭐야?", "제17조"),
    ("해고하려면 얼마나 전에 미리 알려줘야 해?", "제26조"),
    ("연차를 다음 해로 넘길 수 있나요?", "제60조"),
    ("임금은 최소 몇 회 이상 지급해야 해?", "제43조"),
    ("1주 근로시간은 최대 몇 시간까지 가능해?", "제50조"),

    # 식별자 질문
    ("제17조는 무슨 내용이야?", "제17조"),
    ("제23조가 다루는 주제는?", "제23조"),
    ("부당해고 구제신청은 몇 조에 나와 있어?", "제28조"),
    ("제43조의2는 뭐에 관한 조항이야?", "제43조의2"),
    ("근로기준법 제2조는 뭘 정의하고 있어?", "제2조"),
]

for q, a in eval_questions:
    print(f"[{a}] {q}")

[제17조] 근로계약서에 반드시 명시해야 하는 내용은 뭐야?
[제26조] 해고하려면 얼마나 전에 미리 알려줘야 해?
[제60조] 연차를 다음 해로 넘길 수 있나요?
[제43조] 임금은 최소 몇 회 이상 지급해야 해?
[제50조] 1주 근로시간은 최대 몇 시간까지 가능해?
[제17조] 제17조는 무슨 내용이야?
[제23조] 제23조가 다루는 주제는?
[제28조] 부당해고 구제신청은 몇 조에 나와 있어?
[제43조의2] 제43조의2는 뭐에 관한 조항이야?
[제2조] 근로기준법 제2조는 뭘 정의하고 있어?


```
근로기준법의 구조 자체가 조문 단위로 끝남
->이 정도가 한 덩어리로 읽혀야 할 최소 의미 단위라고 정해둔 경계이므로, 그 경계를 그대로 청킹 기준으로 삼음
->자연스러운 경계(##### 제N조)가 있어서 Sliding Window 방식의 overlap이 불필요함
```

---

#### 2. Qdrant 로컬 세팅

In [16]:
# sentence-transformers
from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer("BAAI/bge-m3", device="cuda")

def embed_documents(texts):
    return bge_model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

def embed_query(text):
    return bge_model.encode(text, normalize_embeddings=True, show_progress_bar=False).tolist()

bge_doc_vecs = embed_documents([c["text"] for c in chunks])
print(f"임베딩 차원: {len(bge_doc_vecs[0])}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


임베딩 차원: 1024


In [17]:
try:
    client.close()
    print("기존 client 연결 종료")
except NameError:
    print("기존 client 없음")

기존 client 연결 종료


In [18]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# path를 지정하면 임베디드 모드-해당 폴더에 데이터 저장
client = QdrantClient(path="./qdrant_local")

COLLECTION = "labor_law"

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=len(bge_doc_vecs[0]),
        distance=Distance.COSINE,
    ),
)
# 벡터랑 payload(조항, 원문 텍스트)를 저장
points = [
    PointStruct(
        id=i,
        vector=bge_doc_vecs[i],
        payload={
            "text": chunks[i]["text"],
            "조항": chunks[i]["조항"],
        },
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION, points=points)
print(f"{len(points)}개 포인트 저장 완료")

128개 포인트 저장 완료


In [19]:
# 검색함수
def retrieve(query, top_k=3):
    results = client.query_points(
        collection_name=COLLECTION,
        query=embed_query(query),
        limit=top_k,
    ).points
    return results

In [20]:
!pip install -q --only-binary=:all: bitsandbytes

import bitsandbytes as bnb
print(bnb.__version__)

0.50.2


In [21]:
# 로컬 LLM 로드
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map="cuda",
)

print(f"VRAM 사용량: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

VRAM 사용량: 8.13 GB


In [22]:
def generate(system_prompt, user_prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    output = llm_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

In [67]:
# 로컬 디스크에 실제로 저장되어 있는지 확인
!ls -lh ./qdrant_local/collection/labor_law/
!ls -lh ~/.cache/huggingface/hub/ | grep -i -E "bge|qwen"

total 1.4M
-rw-r--r-- 1 root root 1.4M Sep 29 21:11 storage.sqlite
drwxr-xr-x 6 root root 4.0K Sep 29 21:00 models--BAAI--bge-m3
drwxr-xr-x 6 root root 4.0K Sep 29 21:13 models--Qwen--Qwen2.5-7B-Instruct


In [68]:
# 오프라인에서 캐시된 것만으로 동작하는지 확인
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

answer = rag_answer("근로기준법 제2조는 뭘 정의하고 있어?")
print(answer)

근로기준법 제2조는 법에서 사용하는 주요 용어들을 정의하고 있습니다. 특히 근로자, 사용자, 근로, 근로계약, 임금, 평균임금, 1주, 소정근로시간, 단시간근로자 등의 용어들의 의미를 명확히 정의하였습니다.

근로기준법 제2조의 정의 내용은 다음과 같습니다:

1. 근로자: 사업이나 사업장에 임금을 목적으로 근로를 제공하는 사람.
2. 사용자: 사업주 또는 사업 경영 담당자, 근로자에 대한 사항에 대해 사업주를 위해 행위하는 자.
3. 근로: 정신노동과 육체노동.
4. 근로계약: 근로자가 사용자에게 근로를 제공하고 사용자는 이를 대가로 임금을 지급하는 계약.
5. 임금: 근로의 대가로 근로자에게 지급되는 모든 금품.
6. 평균임금: 근로자가 취업한 후 3개월 미만인 경우에도 해당 3개월 동안의 임금 총액을 그 기간의 총일수로 나눈 금액.
7. 1주: 휴일을 포함한 7일.
8. 소정근로시간: 특정한 주에 제50조제1항의 근로시간을, 특정한 날에 제50조제2항의 근로시간을 초과하여 근로하게 할 수 있는 시간.
9. 단시간근로자: 1주 동안의 소정근로시간이 사업장에서 같은 종류의 업무에 종사하는 통상 근로자의 1주 동안의 소정근로시간에 비해 짧은 근로자.

근로기준법 제2조 제2항은 다음과 같이 정의되어 있습니다:

- 평균임금이 근로자의 통상임금보다 적다면, 통상임금액을 평균임금으로 한다.

따라서 근로기준법 제2조는 근로조건 관련 주요 용어들을 정의하고 있으며, 제2조 제


In [23]:
# 간단히 동작 확인
print(generate("너는 친절한 챗봇이야.", "안녕? 한국어로 짧게 인사해줘."))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos toke

안녕하세요! 좋은 날씨입니다. 무엇을 도와드릴까요?


In [24]:
# RAG 파이프라인
SYSTEM_PROMPT = """당신은 근로기준법 안내 챗봇입니다.
아래 규칙을 지키세요.
1. 제공된 조문 내용에 근거해서만 답변합니다.
2. 답변 끝에 근거 조항을 표기합니다.
3. 조문에 없는 내용이면 "관련 조문을 찾을 수 없습니다"라고 답합니다.
4. 반드시 한국어로만 답변합니다."""

def rag_answer(query, top_k=3):
    hits = retrieve(query, top_k)
    context_text = "\n\n".join(
        f"[{h.payload['조항']}]\n{h.payload['text']}" for h in hits
    )
    user_prompt = f"""다음은 근로기준법에서 검색된 조문입니다.

{context_text}

질문 : {query}"""
    return generate(SYSTEM_PROMPT, user_prompt)

In [25]:
# 테스트
print(rag_answer("연차를 다음 해로 넘길 수 있나요?"))

관련 조문을 찾을 수 없습니다.

이 질문은 근로기준법의 조문에서 직접적으로 다루고 있지 않습니다. 연차를 다음 해로 넘길 수 있는지에 대한 명시적인 규정이 주어진 조문에는 없습니다.


In [26]:
hits = retrieve("연차를 다음 해로 넘길 수 있나요?")
for h in hits:
    print(f"[유사도 {h.score:.3f}] {h.payload['조항']}")
    print(f"  {h.payload['text'][:100]}")

[유사도 0.521] 제21조 (전차금 상계의 금지)
  사용자는 전차금(前借金)이나 그 밖에 근로할 것을 조건으로 하는 전대(前貸)채권과 임금을 상계하지 못한다.
[유사도 0.515] 제85조 (분할보상)
  사용자는 지급 능력이 있는 것을 증명하고 보상을 받는 사람의 동의를 받으면 제80조, 제82조 또는 제84조에 따른 보상금을 1년에 걸쳐 분할보상을 할 수 있다. <개정 2020.
[유사도 0.501] 제57조 (보상 휴가제)
  사용자는 근로자대표와의 서면 합의에 따라 제51조의3, 제52조제2항제2호 및 제56조에 따른 연장근로ㆍ야간근로 및 휴일근로 등에 대하여 임금을 지급하는 것을 갈음하여 휴가를 줄 


In [27]:
info = client.get_collection("labor_law")
print(f"저장된 포인트 수: {info.points_count}")

저장된 포인트 수: 128


In [28]:
# chunks에 실제로 제60조가 있는지, 그리고 몇 번 인덱스인지 확인
for i, c in enumerate(chunks):
    if c["조항"].startswith("제60조") and "제60조의" not in c["조항"]:
        print(i, c["조항"], c["text"][:60])

70 제60조 (연차 유급휴가) **①** 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다. <개정 201


In [31]:
point = client.retrieve(collection_name="labor_law", ids=[70])[0]
print("저장된 payload 조항:", point.payload["조항"])
print("chunks 원본 조항:", chunks[70]["조항"])

저장된 payload 조항: 제60조 (연차 유급휴가)
chunks 원본 조항: 제60조 (연차 유급휴가)


In [32]:
# Hit@3 전체 측정
def measure_hit_at_3(eval_questions, top_k=3):
    correct = 0
    results = []
    for query, answer_article in eval_questions:
        hits = retrieve(query, top_k=top_k)
        retrieved_articles = [h.payload["조항"] for h in hits]
        hit = any(answer_article in a for a in retrieved_articles)
        correct += hit
        results.append({
            "질문": query,
            "정답": answer_article,
            "검색결과": retrieved_articles,
            "Hit": "O" if hit else "X",
        })
    return correct / len(eval_questions), results

hit_rate, results = measure_hit_at_3(eval_questions)

print(f"기준선 Hit@3 = {hit_rate:.1%} ({int(hit_rate*len(eval_questions))}/{len(eval_questions)})\n")
for r in results:
    print(f"[{r['Hit']}] {r['질문']}")
    print(f"  정답: {r['정답']} / 검색됨: {r['검색결과']}")

기준선 Hit@3 = 60.0% (6/10)

[O] 근로계약서에 반드시 명시해야 하는 내용은 뭐야?
  정답: 제17조 / 검색됨: ['제17조 (근로조건의 명시)', '제93조 (취업규칙의 작성ㆍ신고)', '제41조 (근로자의 명부)']
[O] 해고하려면 얼마나 전에 미리 알려줘야 해?
  정답: 제26조 / 검색됨: ['제26조 (해고의 예고)', '제24조 (경영상 이유에 의한 해고의 제한)', '제27조 (해고사유 등의 서면통지)']
[X] 연차를 다음 해로 넘길 수 있나요?
  정답: 제60조 / 검색됨: ['제21조 (전차금 상계의 금지)', '제85조 (분할보상)', '제57조 (보상 휴가제)']
[O] 임금은 최소 몇 회 이상 지급해야 해?
  정답: 제43조 / 검색됨: ['제43조 (임금 지급)', '제60조 (연차 유급휴가)', '제43조의8 (체불 임금등에 대한 손해배상청구)']
[O] 1주 근로시간은 최대 몇 시간까지 가능해?
  정답: 제50조 / 검색됨: ['제50조 (근로시간)', '제69조 (근로시간)', '제51조 (3개월 이내의 탄력적 근로시간제)']
[X] 제17조는 무슨 내용이야?
  정답: 제17조 / 검색됨: ['제19조 (근로조건의 위반)', '제107조 (벌칙)', '제114조 (벌칙)']
[X] 제23조가 다루는 주제는?
  정답: 제23조 / 검색됨: ['제107조 (벌칙)', '제38조 (임금채권의 우선변제)', '제59조 (근로시간 및 휴게시간의 특례)']
[O] 부당해고 구제신청은 몇 조에 나와 있어?
  정답: 제28조 / 검색됨: ['제28조 (부당해고등의 구제신청)', '제30조 (구제명령 등)', '제19조 (근로조건의 위반)']
[X] 제43조의2는 뭐에 관한 조항이야?
  정답: 제43조의2 / 검색됨: ['제107조 (벌칙)', '제110조 (벌칙)', '제113조 (벌칙)']
[O] 근로기준법 제2조는 뭘 정의하고 있어?
  정답: 제2조 / 검색됨: ['제2조 (정의)', '제51조 (3개

#### 기준선 Hit@3 결과

| 질문 | 정답 조항 | 검색된 조항 (Top-3) | Hit |
|---|---|---|---|
| 근로계약서에 반드시 명시해야 하는 내용은 뭐야? | 제17조 | 제17조, 제93조, 제41조 | O |
| 해고하려면 얼마나 전에 미리 알려줘야 해? | 제26조 | 제26조, 제24조, 제27조 | O |
| 연차를 다음 해로 넘길 수 있나요? | 제60조 | 제21조, 제85조, 제57조 | X |
| 임금은 최소 몇 회 이상 지급해야 해? | 제43조 | 제43조, 제60조, 제43조의8 | O |
| 1주 근로시간은 최대 몇 시간까지 가능해? | 제50조 | 제50조, 제69조, 제51조 | O |
| 제17조는 무슨 내용이야? | 제17조 | 제19조, 제107조, 제114조 | X |
| 제23조가 다루는 주제는? | 제23조 | 제107조, 제38조, 제59조 | X |
| 부당해고 구제신청은 몇 조에 나와 있어? | 제28조 | 제28조, 제30조, 제19조 | O |
| 제43조의2는 뭐에 관한 조항이야? | 제43조의2 | 제107조, 제110조, 제113조 | X |
| 근로기준법 제2조는 뭘 정의하고 있어? | 제2조 | 제2조, 제51조, 제1조 | O |

#### 기준선 요약

| 항목 | 값 |
|---|---|
| 전체 Hit@3 | **60.0% (6/10)** |
| 의미 질문 정답률 | 4/5 (80%) |
| 식별자 질문 정답률 | 2/5 (40%) |

#### 실패 분석


| 항목 | 내용 |
|---|---|
| 패턴 A | 반대말 패러프레이징 |
| 실패 사례 | "연차를 다음 해로 넘길 수 있나요?" → 제60조 못 찾음 |
| 원인 | 질문의 "이월"과 조문 원문의 "소멸"이 정반대 방향 표현이라 벡터가 멀어짐. 제60조가 9개 항(①~⑨)을 하나로 뭉친 청크라, 8항 표현이 나머지 항들과 평균화되며 더 희석됨 |
| 근거 | 제60조 본문을 그대로 질의하면 유사도 0.951로 1위 검색되어, 임베딩·Qdrant 파이프라인 자체는 정상임을 확인함 |
| 패턴 B | 조 번호의 이중 등장 혼동 |
| 실패 사례 | "제17조는?", "제23조가 다루는 주제는?", "제43조의2는?" → 제107조·제110조·제113조·제114조(벌칙 조항) 오검색 |
| 원인 | 벌칙 조항(제107·110·114조 등) 본문에 "제7조, 제8조, 제23조제2항, 제36조..." 식으로 수십 개 조 번호가 나열되어 있어, 질문 속 조 번호와 벌칙 조항 속 조 번호 언급이 벡터상으로 구분되지 않음 |
| 근거 | 식별자 질문 5개 중 3개가 이 패턴으로 실패 — 우연이 아니라 구조적 문제 |


#### 해석

패턴 A는 밀집 검색(Dense Retrieval)이 의미는 비슷하지만
표면 어휘가 다른 경우에 약하다는 일반적 한계이고,
패턴 B는 그보다 근로기준법 특유의 구조(벌칙 조항의 조번호 나열)에서 비롯된 문제다.


---

#### 3. 개선하기

3-1. 하이브리드 검색 (BM25 + 벡터, RRF로 통합)

In [33]:
import re

def tokenize(text):
    article_tokens = re.findall(r"제\d+조(?:의\d+)?", text)
    remaining = re.sub(r"제\d+조(?:의\d+)?", " ", text)
    word_tokens = re.findall(r"[가-힣A-Za-z0-9]+", remaining)
    return article_tokens + word_tokens

# 확인
print(tokenize("제23조제2항 또는 제107조를 위반한 자는"))

['제23조', '제107조', '제2항', '또는', '를', '위반한', '자는']


In [35]:
# BM25 인덱스 구축
!pip install -q --only-binary=:all: rank_bm25
from rank_bm25 import BM25Okapi

corpus_tokens = [tokenize(c["조항"] + " " + c["text"]) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)

# 테스트
scores = bm25.get_scores(tokenize("제23조가 다루는 주제는?"))
top5_idx = scores.argsort()[::-1][:5]
for i in top5_idx:
    print(f"[{scores[i]:.2f}] {chunks[i]['조항']}")

[3.62] 제23조 (해고 등의 제한)
[3.27] 제107조 (벌칙)
[3.25] 제29조 (조사 등)
[2.42] 제24조 (경영상 이유에 의한 해고의 제한)
[1.90] 제116조 (과태료)


In [41]:
# Hit@3 재측정
def measure_hit_at_3_hybrid(eval_questions, top_k=3):
    correct = 0
    results = []
    for query, answer_article in eval_questions:
        hits = hybrid_retrieve(query, top_k=top_k)
        retrieved_articles = [h["조항"] for h in hits]
        hit = any(answer_article in a for a in retrieved_articles)
        correct += hit
        results.append({
            "질문": query,
            "정답": answer_article,
            "검색결과": retrieved_articles,
            "Hit": "O" if hit else "X",
        })
    return correct / len(eval_questions), results

hit_rate_hybrid, results_hybrid = measure_hit_at_3_hybrid(eval_questions)

print(f"하이브리드 검색 Hit@3 = {hit_rate_hybrid:.1%} ({int(hit_rate_hybrid*len(eval_questions))}/{len(eval_questions)})")
print(f"기준선 대비: {hit_rate_hybrid - 0.6:+.1%}\n")

for r in results_hybrid:
    print(f"[{r['Hit']}] {r['질문']}")
    print(f"  정답: {r['정답']} / 검색됨: {r['검색결과']}")

하이브리드 검색 Hit@3 = 70.0% (7/10)
기준선 대비: +10.0%

[O] 근로계약서에 반드시 명시해야 하는 내용은 뭐야?
  정답: 제17조 / 검색됨: ['제14조 (법령 주요 내용 등의 게시)', '제17조 (근로조건의 명시)', '제21조 (전차금 상계의 금지)']
[O] 해고하려면 얼마나 전에 미리 알려줘야 해?
  정답: 제26조 / 검색됨: ['제27조 (해고사유 등의 서면통지)', '제24조 (경영상 이유에 의한 해고의 제한)', '제26조 (해고의 예고)']
[X] 연차를 다음 해로 넘길 수 있나요?
  정답: 제60조 / 검색됨: ['제59조 (근로시간 및 휴게시간의 특례)', '제21조 (전차금 상계의 금지)', '제85조 (분할보상)']
[O] 임금은 최소 몇 회 이상 지급해야 해?
  정답: 제43조 / 검색됨: ['제43조 (임금 지급)', '제55조 (휴일)', '제60조 (연차 유급휴가)']
[O] 1주 근로시간은 최대 몇 시간까지 가능해?
  정답: 제50조 / 검색됨: ['제50조 (근로시간)', '제69조 (근로시간)', '제51조 (3개월 이내의 탄력적 근로시간제)']
[X] 제17조는 무슨 내용이야?
  정답: 제17조 / 검색됨: ['제19조 (근로조건의 위반)', '제114조 (벌칙)', '제107조 (벌칙)']
[O] 제23조가 다루는 주제는?
  정답: 제23조 / 검색됨: ['제107조 (벌칙)', '제43조의2 (체불사업주 명단 공개)', '제23조 (해고 등의 제한)']
[O] 부당해고 구제신청은 몇 조에 나와 있어?
  정답: 제28조 / 검색됨: ['제28조 (부당해고등의 구제신청)', '제107조 (벌칙)', '제30조 (구제명령 등)']
[X] 제43조의2는 뭐에 관한 조항이야?
  정답: 제43조의2 / 검색됨: ['제107조 (벌칙)', '제43조의5 (업무위탁 등)', '제43조의6 (체불사업주 명단공개 등을 위한 자료제공 등의 요청)']
[O] 근로기준법 제2조는 뭘 정의하고 있어?

```
개선 1 결과: 60% → 70% (+10%p)
결과 분석: 패턴 B가 일부 해결됨

패턴 B(조 번호 혼동)에서 1건 회복: "제23조가 다루는 주제는?" → 이전엔 제107조가 1위였는데,
이번엔 3위 안에 진짜 제23조가 들음. BM25가 "제23조"라는 정확한 문자열을 정답 조문 제목에서 찾음.

패턴 A(반대말 패러프레이징)는 여전히 실패: "연차를 다음 해로 넘길 수 있나요?"는 그대로 틀림.
BM25도 결국 "이월"이라는 단어 자체가 조문에 없으면 못 찾음. BM25는 키워드 문제(패턴 B)를 풀지,
패러프레이징 문제(패턴 A)는 애초에 겨냥한 게 아님.

패턴 B가 완전히는 안 풀림: "제17조는?", "제43조의2는?"은 여전히 제107조·제114조(벌칙)가 앞섬.
벌칙 조항이 워낙 짧고 조 번호를 빽빽하게 나열하는 구조임.
```

In [42]:
scores = bm25.get_scores(tokenize("제17조는 무슨 내용이야?"))
top5_idx = scores.argsort()[::-1][:5]
for i in top5_idx:
    print(f"[BM25 {scores[i]:.2f}] {chunks[i]['조항']}")

[BM25 3.92] 제19조 (근로조건의 위반)
[BM25 3.58] 제67조 (근로계약)
[BM25 3.33] 제114조 (벌칙)
[BM25 2.77] 제17조 (근로조건의 명시)
[BM25 0.00] 제112조 (고발)


```
BM25는 문서 안에서 드물게 등장하는 단어일수록 가중치를 높게 주는(IDF) 방식임.

"제17조": 조 제목에 한 번만 등장 → 매칭되면 이론상 강한 신호여야 함
"무슨", "내용", "이야": 법률 문서 특성상 이런 일상어는 거의 안 나오거나,
엉뚱하게 다른 조문 본문 어딘가에 우연히 끼어있는 경우도 있음
```

In [43]:
def hybrid_retrieve_v2(query, top_k=3, pool_k=10, rrf_k=60, title_boost=5):
    vec_hits = retrieve(query, top_k=pool_k)
    vec_ranked = [h.payload["조항"] for h in vec_hits]

    bm25_scores = bm25.get_scores(tokenize(query))

    # 질문에 조 번호가 명시적으로 들어있으면, 해당 조항 "제목"에 정확히 일치하는 문서를 강하게 우대
    query_articles = re.findall(r"제\d+조(?:의\d+)?", query)
    if query_articles:
        for i, c in enumerate(chunks):
            title_article = re.match(r"(제\d+조(?:의\d+)?)", c["조항"])
            if title_article and title_article.group(1) in query_articles:
                bm25_scores[i] += title_boost

    bm25_ranked_idx = bm25_scores.argsort()[::-1][:pool_k]
    bm25_ranked = [chunks[i]["조항"] for i in bm25_ranked_idx]

    rrf_score = {}
    for rank, article in enumerate(vec_ranked):
        rrf_score[article] = rrf_score.get(article, 0) + 1 / (rrf_k + rank + 1)
    for rank, article in enumerate(bm25_ranked):
        rrf_score[article] = rrf_score.get(article, 0) + 1 / (rrf_k + rank + 1)

    sorted_articles = sorted(rrf_score.items(), key=lambda x: x[1], reverse=True)
    top_articles = [a for a, _ in sorted_articles[:top_k]]

    article_to_chunk = {c["조항"]: c for c in chunks}
    return [article_to_chunk[a] for a in top_articles]

In [44]:
def measure_hit_at_3_hybrid_v2(eval_questions, top_k=3):
    correct = 0
    results = []
    for query, answer_article in eval_questions:
        hits = hybrid_retrieve_v2(query, top_k=top_k)
        retrieved_articles = [h["조항"] for h in hits]
        hit = any(answer_article in a for a in retrieved_articles)
        correct += hit
        results.append({
            "질문": query,
            "정답": answer_article,
            "검색결과": retrieved_articles,
            "Hit": "O" if hit else "X",
        })
    return correct / len(eval_questions), results

hit_rate_v2, results_v2 = measure_hit_at_3_hybrid_v2(eval_questions)

print(f"제목 부스팅 적용 Hit@3 = {hit_rate_v2:.1%} ({int(hit_rate_v2*len(eval_questions))}/{len(eval_questions)})")
print(f"기준선(60%) 대비: {hit_rate_v2 - 0.6:+.1%}")
print(f"1차 하이브리드(70%) 대비: {hit_rate_v2 - 0.7:+.1%}\n")

for r in results_v2:
    print(f"[{r['Hit']}] {r['질문']}")
    print(f"  정답: {r['정답']} / 검색됨: {r['검색결과']}")

제목 부스팅 적용 Hit@3 = 70.0% (7/10)
기준선(60%) 대비: +10.0%
1차 하이브리드(70%) 대비: +0.0%

[O] 근로계약서에 반드시 명시해야 하는 내용은 뭐야?
  정답: 제17조 / 검색됨: ['제14조 (법령 주요 내용 등의 게시)', '제17조 (근로조건의 명시)', '제21조 (전차금 상계의 금지)']
[O] 해고하려면 얼마나 전에 미리 알려줘야 해?
  정답: 제26조 / 검색됨: ['제27조 (해고사유 등의 서면통지)', '제24조 (경영상 이유에 의한 해고의 제한)', '제26조 (해고의 예고)']
[X] 연차를 다음 해로 넘길 수 있나요?
  정답: 제60조 / 검색됨: ['제59조 (근로시간 및 휴게시간의 특례)', '제21조 (전차금 상계의 금지)', '제85조 (분할보상)']
[O] 임금은 최소 몇 회 이상 지급해야 해?
  정답: 제43조 / 검색됨: ['제43조 (임금 지급)', '제55조 (휴일)', '제60조 (연차 유급휴가)']
[O] 1주 근로시간은 최대 몇 시간까지 가능해?
  정답: 제50조 / 검색됨: ['제50조 (근로시간)', '제69조 (근로시간)', '제51조 (3개월 이내의 탄력적 근로시간제)']
[X] 제17조는 무슨 내용이야?
  정답: 제17조 / 검색됨: ['제19조 (근로조건의 위반)', '제114조 (벌칙)', '제107조 (벌칙)']
[O] 제23조가 다루는 주제는?
  정답: 제23조 / 검색됨: ['제107조 (벌칙)', '제43조의2 (체불사업주 명단 공개)', '제23조 (해고 등의 제한)']
[O] 부당해고 구제신청은 몇 조에 나와 있어?
  정답: 제28조 / 검색됨: ['제28조 (부당해고등의 구제신청)', '제107조 (벌칙)', '제30조 (구제명령 등)']
[X] 제43조의2는 뭐에 관한 조항이야?
  정답: 제43조의2 / 검색됨: ['제107조 (벌칙)', '제43조의5 (업무위탁 등)', '제43조의6 (체불사업주 명단공개 등을 위한 자료제공 등의 요

| 시도 | 변경 내용 | Hit@3 | 기준선 대비 |
|---|---|---|---|
| 기준선 | 벡터 검색만 (BGE-m3) | 60.0% | - |
| 개선 1 | 하이브리드 검색 (BM25 + 벡터, RRF) | 70.0% | +10.0%p |
| 개선 1-보완 | 조 번호 제목 매칭 부스팅 시도 | 70.0% | +10.0%p (효과 없음) |

---

3-2. 부모-자식 청킹

In [48]:
# 조문을 항 단위로 분리
children = []
for parent_id, c in enumerate(chunks):
    parent_text = c["text"]
    sentences = [s.strip() for s in re.split(r"(?<=다\.)\s*", parent_text) if len(s.strip()) > 10]
    for s in sentences:
        children.append({
            "text": s,
            "parent_id": parent_id,
            "parent_text": parent_text,
            "조항": c["조항"],
        })

print(f"부모 청크(조문): {len(chunks)}개")
print(f"자식 청크(문장): {len(children)}개")
print(f"\n예시:", children[100]["text"][:60])

부모 청크(조문): 128개
자식 청크(문장): 412개

예시: 다만, 특별한 사정이 있을 경우에는 당사자 사이의 합의에 의하여 기일을 연장할 수 있다.


In [49]:
# 자식 청크 임베딩 + Qdrant 저장
CHILD_COLLECTION = "labor_law_children"

child_vecs = embed_documents([c["text"] for c in children])

if client.collection_exists(CHILD_COLLECTION):
    client.delete_collection(CHILD_COLLECTION)

client.create_collection(
    collection_name=CHILD_COLLECTION,
    vectors_config=VectorParams(size=len(child_vecs[0]), distance=Distance.COSINE),
)

client.upsert(
    collection_name=CHILD_COLLECTION,
    points=[
        PointStruct(id=i, vector=child_vecs[i], payload=children[i])
        for i in range(len(children))
    ],
)
print(f"{len(children)}개 자식 청크 저장 완료")

412개 자식 청크 저장 완료


In [50]:
# 자식 검색 → 부모 조문으로 확장
def retrieve_parent(query, child_top_k=6, max_parents=3):
    hits = client.query_points(
        collection_name=CHILD_COLLECTION,
        query=embed_query(query),
        limit=child_top_k,
    ).points

    parents = []
    seen = set()
    for h in hits:
        pid = h.payload["parent_id"]
        if pid in seen:
            continue
        seen.add(pid)
        parents.append({
            "조항": h.payload["조항"],
            "parent_text": h.payload["parent_text"],
            "matched_child": h.payload["text"],
            "유사도": h.score,
        })
    return parents[:max_parents]

# 패턴 A 재현 질문으로 먼저 테스트
for p in retrieve_parent("연차를 다음 해로 넘길 수 있나요?"):
    print(f"[{p['조항']}] 유사도 {p['유사도']:.3f}")
    print(f"  걸린 문장: {p['matched_child'][:60]}")

[제33조 (이행강제금)] 유사도 0.526
  걸린 문장: 이 경우 이행강제금은 2년을 초과하여 부과ㆍ징수하지 못한다.
[제21조 (전차금 상계의 금지)] 유사도 0.521
  걸린 문장: 사용자는 전차금(前借金)이나 그 밖에 근로할 것을 조건으로 하는 전대(前貸)채권과 임금을 상계하지 못한다.
[제60조 (연차 유급휴가)] 유사도 0.515
  걸린 문장: <개정 2012.2.1>

**③** 삭제 <2017.11.28>

**④** 사용자는 3년 이상 계속하여 


In [51]:
# 엉뚱한 문장이 3위에 걸림
# 정규식 문제 '-다'로 문장 자름 -> <2017.11.28>같은 개정이력 표시가 문장의 마지막에 위치
# 개정이력 표시를 먼저 제거하고 나서 문장 분리
def clean_amendment_notes(text):
    # <개정 2012.2.1, 2019.1.15> 같은 패턴 제거
    text = re.sub(r"<개정[^>]*>", "", text)
    text = re.sub(r"<신설[^>]*>", "", text)
    text = re.sub(r"<전문개정[^>]*>", "", text)
    return text

children = []
for parent_id, c in enumerate(chunks):
    parent_text = c["text"]
    cleaned = clean_amendment_notes(parent_text)

    # "삭제 <날짜>" 처럼 내용 없는 항은 문장 분리 대상에서 제외
    sentences = [
        s.strip() for s in re.split(r"(?<=다\.)\s*", cleaned)
        if len(s.strip()) > 10 and not s.strip().startswith("삭제")
    ]
    for s in sentences:
        # 볼드 원문자(**①** 등)와 남은 개행/공백 정리
        s_clean = re.sub(r"\*\*[①-⑨]\*\*\s*", "", s).strip()
        if len(s_clean) > 10:
            children.append({
                "text": s_clean,
                "parent_id": parent_id,
                "parent_text": parent_text,
                "조항": c["조항"],
            })

print(f"자식 청크: {len(children)}개")

# 제60조 관련 문장들만 뽑아서 깨끗해졌는지 확인
c60_children = [c for c in children if c["조항"].startswith("제60조 ")]
for c in c60_children:
    print(c["text"][:70])

자식 청크: 375개
사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급
삭제 <2017.11.28>

사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속
이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.
사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 
사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에
다만, 근로자가 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우에는 그 시기를 변경할 수 있다.
제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다.
1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간
  2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 
다만, 사용자의 귀책사유로 사용하지 못한 경우에는 그러하지 아니하다.
사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처우를


In [52]:
# Qdrant 자식 청크 업로드
CHILD_COLLECTION = "labor_law_children"

child_vecs = embed_documents([c["text"] for c in children])

if client.collection_exists(CHILD_COLLECTION):
    client.delete_collection(CHILD_COLLECTION)

client.create_collection(
    collection_name=CHILD_COLLECTION,
    vectors_config=VectorParams(size=len(child_vecs[0]), distance=Distance.COSINE),
)

client.upsert(
    collection_name=CHILD_COLLECTION,
    points=[
        PointStruct(id=i, vector=child_vecs[i], payload=children[i])
        for i in range(len(children))
    ],
)
print(f"{len(children)}개 자식 청크 저장 완료")


375개 자식 청크 저장 완료


In [53]:
# 패턴 A 질문으로 재검색
def retrieve_parent(query, child_top_k=6, max_parents=3):
    hits = client.query_points(
        collection_name=CHILD_COLLECTION,
        query=embed_query(query),
        limit=child_top_k,
    ).points

    parents = []
    seen = set()
    for h in hits:
        pid = h.payload["parent_id"]
        if pid in seen:
            continue
        seen.add(pid)
        parents.append({
            "조항": h.payload["조항"],
            "parent_text": h.payload["parent_text"],
            "matched_child": h.payload["text"],
            "유사도": h.score,
        })
    return parents[:max_parents]

for p in retrieve_parent("연차를 다음 해로 넘길 수 있나요?"):
    print(f"[{p['조항']}] 유사도 {p['유사도']:.3f}")
    print(f"  걸린 문장: {p['matched_child'][:60]}")

[제33조 (이행강제금)] 유사도 0.526
  걸린 문장: 이 경우 이행강제금은 2년을 초과하여 부과ㆍ징수하지 못한다.
[제21조 (전차금 상계의 금지)] 유사도 0.521
  걸린 문장: 사용자는 전차금(前借金)이나 그 밖에 근로할 것을 조건으로 하는 전대(前貸)채권과 임금을 상계하지 못한다.
[제60조 (연차 유급휴가)] 유사도 0.514
  걸린 문장: 삭제 <2017.11.28>

사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1


In [54]:
c60_children = [c for c in children if c["조항"].startswith("제60조 ")]
for c in c60_children:
    print(repr(c["text"][:80]))

'사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
'사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다'
'삭제 <2017.11.28>\n\n사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2'
'이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.'
'사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 때에는 이를 부여하'
'사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에서 정하는 통상임금'
'다만, 근로자가 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우에는 그 시기를 변경할 수 있다.'
'제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다.'
'1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간\n  2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 휴가로 휴업한 기간'
'다만, 사용자의 귀책사유로 사용하지 못한 경우에는 그러하지 아니하다.'
'사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처우를 하여서는 아니 된'


In [55]:
# "삭제 <날짜>" 패턴을 정규식으로 명시적으로 제거
def clean_amendment_notes(text):
    text = re.sub(r"<개정[^>]*>", "", text)
    text = re.sub(r"<신설[^>]*>", "", text)
    text = re.sub(r"<전문개정[^>]*>", "", text)
    # "삭제 <날짜>" 자체를 통째로 제거 (마침표로 안 끝나서 문장 분리에 걸리지 않던 부분)
    text = re.sub(r"삭제\s*<[^>]*>", "", text)
    return text

children = []
for parent_id, c in enumerate(chunks):
    parent_text = c["text"]
    cleaned = clean_amendment_notes(parent_text)
    sentences = [
        s.strip() for s in re.split(r"(?<=다\.)\s*", cleaned)
        if len(s.strip()) > 10
    ]
    for s in sentences:
        s_clean = re.sub(r"\*\*[①-⑨]\*\*\s*", "", s).strip()
        s_clean = re.sub(r"\s+", " ", s_clean)  # 남은 개행/중복 공백도 한 번에 정리
        if len(s_clean) > 10:
            children.append({
                "text": s_clean, "parent_id": parent_id,
                "parent_text": parent_text, "조항": c["조항"],
            })

c60_check = [c for c in children if c["조항"].startswith("제60조 ")]
print("제60조 자식 청크 개수:", len(c60_check))
for c in c60_check:
    print(repr(c["text"][:70]))

제60조 자식 청크 개수: 11
'사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
'사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급'
'사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 '
'이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.'
'사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 '
'사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에'
'다만, 근로자가 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우에는 그 시기를 변경할 수 있다.'
'제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다.'
'1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간 2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 휴가'
'다만, 사용자의 귀책사유로 사용하지 못한 경우에는 그러하지 아니하다.'
'사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처우를'


In [56]:
#Qdrant에 업로드
CHILD_COLLECTION = "labor_law_children"

child_vecs = embed_documents([c["text"] for c in children])

if client.collection_exists(CHILD_COLLECTION):
    client.delete_collection(CHILD_COLLECTION)

client.create_collection(
    collection_name=CHILD_COLLECTION,
    vectors_config=VectorParams(size=len(child_vecs[0]), distance=Distance.COSINE),
)

client.upsert(
    collection_name=CHILD_COLLECTION,
    points=[
        PointStruct(id=i, vector=child_vecs[i], payload=children[i])
        for i in range(len(children))
    ],
)
print(f"{len(children)}개 자식 청크 저장 완료")

373개 자식 청크 저장 완료


In [57]:
def retrieve_parent(query, child_top_k=6, max_parents=3):
    hits = client.query_points(
        collection_name=CHILD_COLLECTION,
        query=embed_query(query),
        limit=child_top_k,
    ).points

    parents = []
    seen = set()
    for h in hits:
        pid = h.payload["parent_id"]
        if pid in seen:
            continue
        seen.add(pid)
        parents.append({
            "조항": h.payload["조항"],
            "parent_text": h.payload["parent_text"],
            "matched_child": h.payload["text"],
            "유사도": h.score,
        })
    return parents[:max_parents]

for p in retrieve_parent("연차를 다음 해로 넘길 수 있나요?"):
    print(f"[{p['조항']}] 유사도 {p['유사도']:.3f}")
    print(f"  걸린 문장: {p['matched_child'][:60]}")

[제33조 (이행강제금)] 유사도 0.526
  걸린 문장: 이 경우 이행강제금은 2년을 초과하여 부과ㆍ징수하지 못한다.
[제21조 (전차금 상계의 금지)] 유사도 0.521
  걸린 문장: 사용자는 전차금(前借金)이나 그 밖에 근로할 것을 조건으로 하는 전대(前貸)채권과 임금을 상계하지 못한다.
[제85조 (분할보상)] 유사도 0.500
  걸린 문장: 사용자는 지급 능력이 있는 것을 증명하고 보상을 받는 사람의 동의를 받으면 제80조, 제82조 또는 제84조


In [58]:
# "소멸"이라는 단어가 들어간 진짜 원칙 문장이 children에 있는지 확인
소멸_children = [c for c in children if "소멸" in c["text"]]
for c in 소멸_children:
    print(c["조항"], ":", c["text"][:80])

제44조의3 (건설업의 공사도급에 있어서의 임금에 관한 특례) : 직상 수급인 또는 원수급인이 제1항 및 제2항에 따라 하수급인이 사용한 근로자에게 임금에 해당하는 금액을 지급한 경우에는 하수급인에 대한 하도급
제49조 (임금의 시효) : 이 법에 따른 임금채권은 3년간 행사하지 아니하면 시효로 소멸한다.
제60조 (연차 유급휴가) : 1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간 2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 휴가로 휴업한 기간 3
제61조 (연차 유급휴가의 사용 촉진) : 사용자가 제60조제1항ㆍ제2항 및 제4항에 따른 유급휴가(계속하여 근로한 기간이 1년 미만인 근로자의 제60조제2항에 따른 유급휴가는 제외한다)
제61조 (연차 유급휴가의 사용 촉진) : 1. 제60조제8항 본문에 따른 기간이 끝나기 6개월 전을 기준으로 10일 이내에 사용자가 근로자별로 사용하지 아니한 휴가 일수를 알려주고, 근
제91조 (서류의 보존) : 사용자는 재해보상에 관한 중요한 서류를 재해보상이 끝나지 아니하거나 제92조에 따라 재해보상 청구권이 시효로 소멸되기 전에 폐기하여서는 아니 된
제92조 (시효) : 이 법의 규정에 따른 재해보상 청구권은 3년간 행사하지 아니하면 시효로 소멸한다.


In [59]:
idx60 = [i for i, c in enumerate(chunks) if c["조항"].startswith("제60조 ")][0]
print(chunks[idx60]["text"])

**①** 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다. <개정 2012.2.1>

**②** 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다. <개정 2012.2.1>

**③** 삭제 <2017.11.28>

**④** 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 주어야 한다. 이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.

**⑤** 사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 때에는 이를 부여하여야 한다. <신설 2026.6.9>

**⑥** 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에서 정하는 통상임금 또는 평균임금을 지급하여야 한다. 다만, 근로자가 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우에는 그 시기를 변경할 수 있다. <개정 2026.6.9>

**⑦** 제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다. <개정 2012.2.1, 2017.11.28, 2024.10.22, 2026.2.19, 2026.6.9>

  1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간
  2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 휴가로 휴업한 기간
  3. 「남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률」 제19조에 따른 육아휴직으로 휴업한 기간
  4. 「남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률」 제19조의2제1항에 따른 육아기 근로시간 단축을 사용하여 단축된 근로시간
  5. 제74조제7항에 따른 임신기 근로시간 단축을 사용하여 단축된 근로

In [60]:
cleaned_60 = clean_amendment_notes(chunks[idx60]["text"])
sentences_60 = re.split(r"(?<=다\.)\s*", cleaned_60)

for i, s in enumerate(sentences_60):
    print(f"--- 조각 {i} (길이 {len(s)}) ---")
    print(repr(s[:150]))

--- 조각 0 (길이 52) ---
'**①** 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
--- 조각 1 (길이 87) ---
'**②** 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다.'
--- 조각 2 (길이 105) ---
'**③** \n\n**④** 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 주어야 한다.'
--- 조각 3 (길이 36) ---
'이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.'
--- 조각 4 (길이 92) ---
'**⑤** 사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 때에는 이를 부여하여야 한다.'
--- 조각 5 (길이 105) ---
'**⑥** 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에서 정하는 통상임금 또는 평균임금을 지급하여야 한다.'
--- 조각 6 (길이 65) ---
'다만, 근로자가 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우에는 그 시기를 변경할 수 있다.'
--- 조각 7 (길이 60) ---
'**⑦** 제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다.'
--- 조각 8 (길이 370) ---
'1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간\n  2. 임신 중의 여성이 제74조제1항부터 제3항까지의 규정에 따른 휴가로 휴업한 기간\n  3. 「남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률」 제19조에 따른 육아휴직으로 휴업한 기간\n  4. 「남녀고용평등과'
--- 조각 9 (길이 38) ---
'다만

| 항목 | 내용 |
|---|---|
| 시도 | 조문을 "다." 기준으로 문장 분리해 자식 청크 생성, 검색 후 부모 조문으로 확장 |
| 목표 | 제60조 8항이 조문 전체에 희석되는 문제(패턴 A) 해결 |
| 결과 | Hit@3 개선 실패, 제60조가 Top-3에서 완전히 사라짐 |
| 원인 1 | ⑦항 호(1~5) 리스트가 "다."로 안 끝나 다음 문장까지 흡수됨 |
| 원인 2 | 원칙("소멸된다")은 리스트와 뒤섞이고 예외 문장만 홀로 분리됨 |
| 해석 | 법령은 괄호·호 리스트·다양한 종결어미로 사내규정보다 불규칙함 |
| 개선 방향 | "다." 대신 항 번호(①~⑨) 자체를 1차 분리 기준으로 삼아야 함 |

---

#### 3-3 부모-자식 청킹: 항번호 기준으로 청킹

In [61]:
JANG_MARKS = "①②③④⑤⑥⑦⑧⑨⑩"

def split_by_hang(parent_text):
    cleaned = clean_amendment_notes(parent_text)
    cleaned = re.sub(r"\*\*([①-⑩])\*\*", r"\1", cleaned)  # 볼드 마크 제거하고 원문자만 남김

    pattern = f"(?=[{JANG_MARKS}])"
    pieces = re.split(pattern, cleaned)
    pieces = [p.strip() for p in pieces if p.strip()]

    return pieces

# 제60조로 테스트
idx60 = [i for i, c in enumerate(chunks) if c["조항"].startswith("제60조 ")][0]
pieces_60 = split_by_hang(chunks[idx60]["text"])

for p in pieces_60:
    print(repr(p[:80]))
    print("-")

'① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
-
'② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 '
-
'③'
-
'④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 '
-
'⑤ 사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 때에는 이를 부'
-
'⑥ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에서 정하는 통상'
-
'⑦ 제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다. \n\n  1. 근로자가 업무상의 부상 또는 '
-
'⑧ 제1항ㆍ제2항 및 제4항에 따른 휴가는 1년간(계속하여 근로한 기간이 1년 미만인 근로자의 제2항에 따른 유급휴가는 최초 1년의 근로가 끝날'
-
'⑨ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처우를 하여서는 아니'
-


In [62]:
# 자식 청크 생성
children_v2 = []
for parent_id, c in enumerate(chunks):
    pieces = split_by_hang(c["text"])
    for piece in pieces:
        if len(piece) > 10:
            children_v2.append({
                "text": piece,
                "parent_id": parent_id,
                "parent_text": c["text"],
                "조항": c["조항"],
            })

print(f"항 단위 자식 청크: {len(children_v2)}개")

c60_v2 = [c for c in children_v2 if c["조항"].startswith("제60조 ")]
for c in c60_v2:
    print(repr(c["text"][:70]))

항 단위 자식 청크: 277개
'① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
'② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 '
'④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하'
'⑤ 사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구'
'⑥ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 '
'⑦ 제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다. \n\n  1. 근로자가 업'
'⑧ 제1항ㆍ제2항 및 제4항에 따른 휴가는 1년간(계속하여 근로한 기간이 1년 미만인 근로자의 제2항에 따른 유급휴가는 최초 '
'⑨ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처'


In [63]:
# Qdrant 업로드 + 재검색
CHILD_COLLECTION_V2 = "labor_law_children_hang"

child_vecs_v2 = embed_documents([c["text"] for c in children_v2])

if client.collection_exists(CHILD_COLLECTION_V2):
    client.delete_collection(CHILD_COLLECTION_V2)

client.create_collection(
    collection_name=CHILD_COLLECTION_V2,
    vectors_config=VectorParams(size=len(child_vecs_v2[0]), distance=Distance.COSINE),
)

client.upsert(
    collection_name=CHILD_COLLECTION_V2,
    points=[
        PointStruct(id=i, vector=child_vecs_v2[i], payload=children_v2[i])
        for i in range(len(children_v2))
    ],
)
print(f"{len(children_v2)}개 저장 완료")

def retrieve_parent_v2(query, child_top_k=6, max_parents=3):
    hits = client.query_points(
        collection_name=CHILD_COLLECTION_V2,
        query=embed_query(query),
        limit=child_top_k,
    ).points

    parents, seen = [], set()
    for h in hits:
        pid = h.payload["parent_id"]
        if pid in seen:
            continue
        seen.add(pid)
        parents.append({
            "조항": h.payload["조항"],
            "parent_text": h.payload["parent_text"],
            "matched_child": h.payload["text"],
            "유사도": h.score,
        })
    return parents[:max_parents]

for p in retrieve_parent_v2("연차를 다음 해로 넘길 수 있나요?"):
    print(f"[{p['조항']}] 유사도 {p['유사도']:.3f}")
    print(f"  걸린 항: {p['matched_child'][:70]}")

277개 저장 완료
[제21조 (전차금 상계의 금지)] 유사도 0.521
  걸린 항: 사용자는 전차금(前借金)이나 그 밖에 근로할 것을 조건으로 하는 전대(前貸)채권과 임금을 상계하지 못한다.
[제85조 (분할보상)] 유사도 0.500
  걸린 항: 사용자는 지급 능력이 있는 것을 증명하고 보상을 받는 사람의 동의를 받으면 제80조, 제82조 또는 제84조에 따른 보상금을 
[제62조 (유급휴가의 대체)] 유사도 0.497
  걸린 항: 사용자는 근로자대표와의 서면 합의에 따라 제60조에 따른 연차 유급휴가일을 갈음하여 특정한 근로일에 근로자를 휴무시킬 수 있다


In [64]:
c60_v2 = [c for c in children_v2 if c["조항"].startswith("제60조 ")]
for c in c60_v2:
    print(repr(c["text"][:100]))

'① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.'
'② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다.'
'④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 주어야 한다. 이 경우 가'
'⑤ 사용자는 근로자가 제1항ㆍ제2항 및 제4항에 따른 유급휴가를 대통령령으로 정하는 시간단위 및 일수의 범위에서 분할하여 청구한 때에는 이를 부여하여야 한다.'
'⑥ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 휴가를 근로자가 청구한 시기에 주어야 하고, 그 기간에 대하여는 취업규칙 등에서 정하는 통상임금 또는 평균임금을 지급하여야 한다'
'⑦ 제1항 및 제2항을 적용하는 경우 다음 각 호의 어느 하나에 해당하는 기간은 출근한 것으로 본다. \n\n  1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간\n  2. 임신 '
'⑧ 제1항ㆍ제2항 및 제4항에 따른 휴가는 1년간(계속하여 근로한 기간이 1년 미만인 근로자의 제2항에 따른 유급휴가는 최초 1년의 근로가 끝날 때까지의 기간을 말한다) 행사하지 '
'⑨ 사용자는 제1항ㆍ제2항ㆍ제4항 및 제5항에 따른 유급휴가의 청구 또는 사용을 이유로 근로자를 해고하거나 그 밖의 불리한 처우를 하여서는 아니 된다.'


In [65]:
eighth_hang = [c for c in c60_v2 if "소멸된다" in c["text"]]
if eighth_hang:
    print("8항 텍스트:", eighth_hang[0]["text"])

    # 이 문장을 그대로 쿼리로 넣었을 때 자기 자신이 1위로 나오는지
    hits = client.query_points(
        collection_name=CHILD_COLLECTION_V2,
        query=embed_query(eighth_hang[0]["text"][:200]),
        limit=3,
    ).points
    for h in hits:
        print(f"[{h.score:.3f}] {h.payload['조항']}")
else:
    print("8항(소멸된다 포함)을 찾을 수 없습니다 — 파싱 문제")

8항 텍스트: ⑧ 제1항ㆍ제2항 및 제4항에 따른 휴가는 1년간(계속하여 근로한 기간이 1년 미만인 근로자의 제2항에 따른 유급휴가는 최초 1년의 근로가 끝날 때까지의 기간을 말한다) 행사하지 아니하면 소멸된다. 다만, 사용자의 귀책사유로 사용하지 못한 경우에는 그러하지 아니하다.
[1.000] 제60조 (연차 유급휴가)
[0.807] 제61조 (연차 유급휴가의 사용 촉진)
[0.799] 제61조 (연차 유급휴가의 사용 촉진)


#### 부모-자식 청킹 시도 결과 (v1: 문장 단위 → v2: 항 단위로 재시도)

| 항목 | 내용 |
|---|---|
| v1 시도 | "다." 기준 문장 분리 |
| v1 결과 | 실패 — 8항이 리스트에 흡수되어 원칙과 예외가 분리됨 |
| v2 시도 | 항 번호(①~⑨) 기준 분리로 v1의 원인 해결 |
| v2 결과 | 청킹 품질은 완벽(자기참조 유사도 1.000)했으나 Hit 실패는 동일 |
| 최종 원인 | 청킹 문제가 아니라 BGE 임베딩이 "이월"(질문 어휘)과 "소멸"(조문 어휘)이라는 반의어 관계를 충분히 가깝게 인코딩하지 못함 |
| 해석 | 이 실패는 검색 구조(청킹)로는 해결 불가능한, 임베딩 모델의 의미 표현 한계임. Query Rewriting(질문을 "휴가가 소멸되나요?"처럼 조문 어휘에 가깝게 재작성)이나 리랭커가 더 적합한 해결책 |

###

#### 개선책
| 방법 | 지금 문제 해결 가능성 | 비용 |
|---|---|---|
| 의미단위 청킹 (더 정교하게) | 낮음 — 이미 청크는 완벽함 | 낮음 |
| Query Rewrite | 높음 — 어휘 자체를 조문식으로 변환 | LLM 호출 1회 추가 |
| HyDE | 높음 — 반의어 문제를 가상 문서로 우회 | LLM 호출 1회 추가 (더 김) |
| 동의어 사전 확장 | 중간 — 사전에 없는 패턴은 못 잡음 | 낮음, 수작업 필요 |

####

3-3. Query Rewrite: 법률 조문 어휘로 질문을 바꾸도록 함

In [69]:
# Query Rewrite 구현
def rewrite_query_v2(query):
    prompt = f"""사용자의 일상적인 질문을, 그 질문이 법적으로 다뤄질 때 쓰이는 핵심 법률 용어와 개념으로 재구성하세요.

지침:
- 사용자가 쓴 단어를 그대로 옮기지 말고, 그 단어가 법 조문에서는 어떤 용어로 표현될지 생각해서 바꾸세요.
- 예를 들어 "이월하다", "넘기다"는 조문에서 "소멸되지 않는다" 또는 "사용 기한"처럼 표현될 수 있습니다.
- 질문의 실제 의도(허용되는지, 예외가 있는지)는 유지하되, 표현은 완전히 법률적으로 바꾸세요.
- 재구성한 질문만 출력하세요.

질문: {query}
법률 용어로 재구성:"""
    return generate("당신은 법률 상담 접수원입니다. 고객의 질문을 변호사가 조문을 찾기 쉬운 표현으로 바꿔줍니다.", prompt, max_new_tokens=80).strip()

In [70]:
original = "연차를 다음 해로 넘길 수 있나요?"
rewritten_v2 = rewrite_query_v2(original)

print("원본:", original)
print("변환:", rewritten_v2)

hits = retrieve(rewritten_v2, top_k=3)
for h in hits:
    print(h.payload["조항"])

원본: 연차를 다음 해로 넘길 수 있나요?
변환: 연차를 다음 해로 소멸하지 않게 처리할 수 있나요?
제91조 (서류의 보존)
제92조 (시효)
제86조 (보상 청구권)


```
1차 문제(원본 질문):  "이월"이라는 단어가 조문에 없어서 검색 실패
2차 문제(재작성 후):  "소멸"이라는 단어는 생겼지만, 그 단어가 여러 조문에 공통으로
                      등장해서 오히려 "동음이의어스러운" 혼란이 생김
```

In [71]:
def measure_hit_at_3_rewrite(eval_questions, top_k=3):
    correct = 0
    results = []
    for query, answer_article in eval_questions:
        rewritten = rewrite_query_v2(query)
        hits = retrieve(rewritten, top_k=top_k)
        retrieved_articles = [h.payload["조항"] for h in hits]
        hit = any(answer_article in a for a in retrieved_articles)
        correct += hit
        results.append({
            "원본질문": query,
            "재작성": rewritten,
            "정답": answer_article,
            "검색결과": retrieved_articles,
            "Hit": "O" if hit else "X",
        })
    return correct / len(eval_questions), results

hit_rate_rw, results_rw = measure_hit_at_3_rewrite(eval_questions)

print(f"Query Rewrite 적용 Hit@3 = {hit_rate_rw:.1%}")
print(f"기준선(60%) 대비: {hit_rate_rw - 0.6:+.1%}\n")

for r in results_rw:
    print(f"[{r['Hit']}] {r['원본질문']}")
    print(f"    → {r['재작성']}")
    print(f"    정답: {r['정답']} / 검색됨: {r['검색결과']}")

Query Rewrite 적용 Hit@3 = 60.0%
기준선(60%) 대비: +0.0%

[O] 근로계약서에 반드시 명시해야 하는 내용은 뭐야?
    → 근로계약서에 반드시 명시되어야 하는 내용은 무엇인가요?
    정답: 제17조 / 검색됨: ['제17조 (근로조건의 명시)', '제93조 (취업규칙의 작성ㆍ신고)', '제41조 (근로자의 명부)']
[O] 해고하려면 얼마나 전에 미리 알려줘야 해?
    → 해고를 위한 통지의무는 몇 달이나 되어야 하는지?
    정답: 제26조 / 검색됨: ['제26조 (해고의 예고)', '제27조 (해고사유 등의 서면통지)', '제28조 (부당해고등의 구제신청)']
[X] 연차를 다음 해로 넘길 수 있나요?
    → 연차를 다음 해로 소멸하지 않게 처리할 수 있나요?
    정답: 제60조 / 검색됨: ['제91조 (서류의 보존)', '제92조 (시효)', '제86조 (보상 청구권)']
[O] 임금은 최소 몇 회 이상 지급해야 해?
    → 임금은 최소 몇 회 이상 지급해야 하는지에 대한 질문은, 실제로는 임금 지급 주기를 다루는 것이므로, 법적으로는 "임금 지급 주기" 또는 "임금 지급 주기의 최소 횟수"라는 용어로 표현될 수 있습니다.
    정답: 제43조 / 검색됨: ['제43조 (임금 지급)', '제2조 (정의)', '제43조의8 (체불 임금등에 대한 손해배상청구)']
[O] 1주 근로시간은 최대 몇 시간까지 가능해?
    → 1주간의 근로 시간은 최대 몇 시간까지 가능하며, 이는 어떠한 특별한 예외 조건 없이 적용됩니까?
    정답: 제50조 / 검색됨: ['제50조 (근로시간)', '제51조 (3개월 이내의 탄력적 근로시간제)', '제51조의2 (3개월을 초과하는 탄력적 근로시간제)']
[X] 제17조는 무슨 내용이야?
    → 제17조는 어떤 권리가 시간 경과로 소멸되지 않는다는 내용을 담고 있습니다.
    정답: 제17조 / 검색됨: ['제86조 (보상 청구권)', '제49조 (임금의 시효)

| 항목 | 내용 |
|---|---|
| 시도 | Query Rewrite — LLM이 질문을 법률 용어로 재구성 후 검색 |
| 결과 | Hit@3 60% (기준선과 동일), 그러나 내부 구성은 달라짐 |
| 긍정적 신호 | "이월→소멸" 같은 반의어 연결을 LLM이 성공적으로 수행함 |
| 부작용 (핵심 발견) | 식별자 질문에서 LLM이 모르는 조 번호의 내용을 환각으로 지어내 재작성함 → 원본 질문의 단서(조 번호)까지 왜곡되어 오히려 위험한 실패 유형 생성 |
| 해석 | Query Rewrite는 "의미를 몰라서 묻는 의미 질문"에는 적합하지만, "내용 자체를 확인하려는 식별자 질문"에는 LLM이 답을 지어낼 위험이 있어 부적합( 질문 유형별로 재작성 적용 여부를 나누는 게 필요) |

In [73]:
# 식별자 질문 개선
def smart_retrieve(query, top_k=3):
    # 질문에 조 번호가 명시돼 있으면 재작성 없이 하이브리드 검색 그대로 사용
    has_article_number = bool(re.search(r"제\d+조", query))
    if has_article_number:
        return hybrid_retrieve(query, top_k=top_k)
    else:
        rewritten = rewrite_query_v2(query)
        return hybrid_retrieve(rewritten, top_k=top_k)

In [74]:
def measure_hit_at_3_smart(eval_questions, top_k=3):
    correct = 0
    results = []
    for query, answer_article in eval_questions:
        hits = smart_retrieve(query, top_k=top_k)
        retrieved_articles = [h["조항"] for h in hits]
        hit = any(answer_article in a for a in retrieved_articles)
        correct += hit
        results.append({
            "질문": query, "정답": answer_article,
            "검색결과": retrieved_articles, "Hit": "O" if hit else "X",
        })
    return correct / len(eval_questions), results

hit_rate_smart, results_smart = measure_hit_at_3_smart(eval_questions)

print(f"하이브리드 + 조건부 재작성 Hit@3 = {hit_rate_smart:.1%}")
print(f"기준선(60%) 대비: {hit_rate_smart - 0.6:+.1%}\n")

for r in results_smart:
    print(f"[{r['Hit']}] {r['질문']} → {r['검색결과']}")

하이브리드 + 조건부 재작성 Hit@3 = 70.0%
기준선(60%) 대비: +10.0%

[O] 근로계약서에 반드시 명시해야 하는 내용은 뭐야? → ['제14조 (법령 주요 내용 등의 게시)', '제17조 (근로조건의 명시)', '제21조 (전차금 상계의 금지)']
[O] 해고하려면 얼마나 전에 미리 알려줘야 해? → ['제26조 (해고의 예고)', '제24조 (경영상 이유에 의한 해고의 제한)', '제27조 (해고사유 등의 서면통지)']
[X] 연차를 다음 해로 넘길 수 있나요? → ['제91조 (서류의 보존)', '제59조 (근로시간 및 휴게시간의 특례)', '제92조 (시효)']
[O] 임금은 최소 몇 회 이상 지급해야 해? → ['제43조 (임금 지급)', '제45조 (비상시 지급)', '제36조 (금품 청산)']
[O] 1주 근로시간은 최대 몇 시간까지 가능해? → ['제52조 (선택적 근로시간제)', '제53조 (연장 근로의 제한)', '제50조 (근로시간)']
[X] 제17조는 무슨 내용이야? → ['제19조 (근로조건의 위반)', '제114조 (벌칙)', '제107조 (벌칙)']
[O] 제23조가 다루는 주제는? → ['제107조 (벌칙)', '제43조의2 (체불사업주 명단 공개)', '제23조 (해고 등의 제한)']
[O] 부당해고 구제신청은 몇 조에 나와 있어? → ['제30조 (구제명령 등)', '제104조 (감독 기관에 대한 신고)', '제28조 (부당해고등의 구제신청)']
[X] 제43조의2는 뭐에 관한 조항이야? → ['제107조 (벌칙)', '제43조의5 (업무위탁 등)', '제43조의6 (체불사업주 명단공개 등을 위한 자료제공 등의 요청)']
[O] 근로기준법 제2조는 뭘 정의하고 있어? → ['제2조 (정의)', '제44조의2 (건설업에서의 임금 지급 연대책임)', '제51조 (3개월 이내의 탄력적 근로시간제)']


#### 개선 실험 전체 결과

| 시도 | 변경 내용 | Hit@3 | 기준선 대비 |
|---|---|---|---|
| 기준선 | 벡터 검색만 (BGE-m3) | 60.0% | - |
| 개선 1 | 하이브리드 검색 (BM25 + 벡터, RRF) | 70.0% | +10.0%p |
| 개선 2-a | 부모-자식 청킹 (문장 단위, "다." 기준) | - | 효과 없음, 원인 분석 |
| 개선 2-b | 부모-자식 청킹 (항 단위, ①~⑨ 기준) | - | 효과 없음, 청킹 문제 아님을 확정 |
| 개선 3 | Query Rewrite (전체 질문에 적용) | 60.0% | +0.0%p, 새로운 부작용(환각) 발견 |
| 개선 4 | 하이브리드 + 조건부 Query Rewrite | 70.0% | +10.0%p, 부작용 제거 |

---

#### 결론 및 정리

```
문서(근로기준법 md) → 조문 단위 파싱(128개)
        │
        ▼
BGE-m3 임베딩 (로컬, sentence-transformers) → 1024차원 벡터
        │
        ▼
Qdrant (파일 기반 임베디드 모드, ./qdrant_local)
        │
        ▼
검색(top-3) → Qwen2.5-7B-Instruct (4bit 양자화, 로컬 GPU 추론)
        │
        ▼
답변 생성 (출처 조항 표기, 한국어 강제 지시 포함 시스템 프롬프트)
```

| 시도 | 변경 내용 | Hit@3 | 기준선 대비 |
|---|---|---|---|
| 기준선 | 벡터 검색만 (BGE-m3) | 60.0% | - |
| 개선 1 | 하이브리드 검색 (BM25 + 벡터, RRF) | 70.0% | +10.0%p |
| 개선 2-a | 부모-자식 청킹 (문장 단위, "다." 기준 분리) | - | 효과 없음 |
| 개선 2-b | 부모-자식 청킹 (항 단위, ①~⑨ 기준 분리) | - | 효과 없음 |
| 개선 3 | Query Rewrite (전체 질문에 적용) | 60.0% | +0.0%p |
| 개선 4 | 하이브리드 검색 + 조건부 Query Rewrite | 70.0% | +10.0%p |

```
이번 실험에서 Hit@3를 실질적으로 개선한 유일한 단일 기법은 하이브리드 검색이며(60%→70%),
이는 근로기준법처럼 정확한 식별자(조 번호) 검색이 중요한 도메인에서 키워드 매칭의 가치를 실증적으로 보여줌.

부모-자식 청킹과 Query Rewrite는 최종 수치 개선에는 실패했지만,
각각 임베딩 모델의 의미 표현 한계와 LLM 재작성의 환각 위험이라는 이 RAG 시스템의 구조적 한계를 구체적으로 규명하는 성과를 거두었음.

특히 하이브리드 검색과 조건부 Query Rewrite를 결합한
최종 파이프라인은 Hit@3 수치는 유지하면서도 답변의 신뢰성을 개선했다는 점에서,
단일 지표로 환원되지 않는 실질적 개선을 이뤘다고 평가함.
```